# Análisis de Animales Identificados con Microchip por Localidad en Bogotá

## Proyecto: Georreferenciación y Análisis de Datos Urbanos

**Objetivo:** Realizar un análisis ETL completo de los datos de animales identificados con microchip en Bogotá, generando visualizaciones interactivas que permitan identificar patrones por localidad.

**Dataset:** Animales identificados con microchip por localidad (Datos Abiertos Bogotá)

---

## 1. Importar Librerías Necesarias

In [ ]:
# Procesamiento de datos
import pandas as pd
import numpy as np

# Visualización
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Mapas interactivos
import folium
from folium import plugins

# Base de datos
import sqlite3
from pathlib import Path

# Configuración
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Librerías importadas correctamente")

## 2. EXTRACT - Extracción de Datos

In [ ]:
# Cargar datos
file_path = '../data/c4p-animales-identificados-con-microship-por-localidad.csv'

# Leer CSV con encoding adecuado
df = pd.read_csv(file_path, sep=';', encoding='latin-1')

print(f"📊 Dataset cargado: {len(df)} registros")
print(f"📋 Columnas: {list(df.columns)}")
print(f"\n🔍 Primeras filas:")
df.head(10)

In [ ]:
# Información del dataset
print("📈 Información del Dataset:\n")
print(df.info())
print("\n" + "="*50 + "\n")
print("📊 Estadísticas Descriptivas:\n")
print(df.describe(include='all'))

## 3. TRANSFORM - Limpieza y Transformación de Datos

In [ ]:
# Verificar valores nulos
print("🔍 Valores nulos por columna:\n")
print(df.isnull().sum())
print(f"\n📊 Total de valores nulos: {df.isnull().sum().sum()}")

# Eliminar duplicados
duplicados_antes = len(df)
df = df.drop_duplicates(subset='microchip_animal')
duplicados_despues = len(df)
print(f"\n🗑️ Duplicados eliminados: {duplicados_antes - duplicados_despues}")

# Limpiar espacios en blanco
df['localidad_territorializacion'] = df['localidad_territorializacion'].str.strip().str.upper()
df['especie'] = df['especie'].str.strip().str.upper()
df['sexo_animal'] = df['sexo_animal'].str.strip().str.upper()
df['tamano_animal'] = df['tamano_animal'].str.strip().str.upper()
df['potencialmente_peligroso'] = df['potencialmente_peligroso'].str.strip().str.upper()

print("\n✅ Datos limpiados correctamente")

In [ ]:
# Análisis de categorías
print("📊 ANÁLISIS DE CATEGORÍAS\n")
print("="*50)

print("\n🏘️ Localidades:")
print(df['localidad_territorializacion'].value_counts())

print("\n🐾 Especies:")
print(df['especie'].value_counts())

print("\n⚧ Sexo:")
print(df['sexo_animal'].value_counts())

print("\n📏 Tamaño:")
print(df['tamano_animal'].value_counts())

print("\n⚠️ Potencialmente peligroso:")
print(df['potencialmente_peligroso'].value_counts())

In [ ]:
# Coordenadas de las localidades de Bogotá (centroide aproximado)
coordenadas_localidades = {
    'USAQUEN': {'lat': 4.6944, 'lon': -74.0306},
    'CHAPINERO': {'lat': 4.6308, 'lon': -74.0658},
    'SANTA FE': {'lat': 4.6053, 'lon': -74.0755},
    'SAN CRISTOBAL': {'lat': 4.5653, 'lon': -74.0825},
    'USME': {'lat': 4.4825, 'lon': -74.1284},
    'TUNJUELITO': {'lat': 4.5748, 'lon': -74.1329},
    'BOSA': {'lat': 4.6186, 'lon': -74.1878},
    'KENNEDY': {'lat': 4.6281, 'lon': -74.1550},
    'FONTIBON': {'lat': 4.6728, 'lon': -74.1444},
    'ENGATIVA': {'lat': 4.7011, 'lon': -74.1122},
    'SUBA': {'lat': 4.7565, 'lon': -74.0826},
    'BARRIOS UNIDOS': {'lat': 4.6639, 'lon': -74.0819},
    'TEUSAQUILLO': {'lat': 4.6417, 'lon': -74.0875},
    'LOS MARTIRES': {'lat': 4.6117, 'lon': -74.0939},
    'ANTONIO NARINO': {'lat': 4.5850, 'lon': -74.1111},
    'PUENTE ARANDA': {'lat': 4.6144, 'lon': -74.1194},
    'LA CANDELARIA': {'lat': 4.5964, 'lon': -74.0739},
    'RAFAEL URIBE URIBE': {'lat': 4.5528, 'lon': -74.1153},
    'CIUDAD BOLIVAR': {'lat': 4.5753, 'lon': -74.1772},
    'SUMAPAZ': {'lat': 4.2417, 'lon': -74.2481}
}

# Crear DataFrame de localidades
df_localidades = pd.DataFrame([
    {'localidad': k, 'latitud': v['lat'], 'longitud': v['lon']}
    for k, v in coordenadas_localidades.items()
])

print("🗺️ Coordenadas de localidades cargadas:")
print(df_localidades)

In [ ]:
# Agregar datos por localidad
df_agregado = df.groupby('localidad_territorializacion').agg({
    'microchip_animal': 'count',
    'especie': lambda x: x.value_counts().index[0],  # Especie más común
}).reset_index()

df_agregado.columns = ['localidad', 'total_animales', 'especie_predominante']

# Agregar estadísticas adicionales
# Caninos por localidad
caninos = df[df['especie'] == 'CANINO'].groupby('localidad_territorializacion').size()
felinos = df[df['especie'] == 'FELINO'].groupby('localidad_territorializacion').size()
peligrosos = df[df['potencialmente_peligroso'] == 'SI'].groupby('localidad_territorializacion').size()

df_agregado['total_caninos'] = df_agregado['localidad'].map(caninos).fillna(0).astype(int)
df_agregado['total_felinos'] = df_agregado['localidad'].map(felinos).fillna(0).astype(int)
df_agregado['total_peligrosos'] = df_agregado['localidad'].map(peligrosos).fillna(0).astype(int)

# Merge con coordenadas
df_mapa = df_agregado.merge(df_localidades, on='localidad', how='left')

# Eliminar localidades sin coordenadas
df_mapa = df_mapa.dropna(subset=['latitud', 'longitud'])

print("📊 Dataset agregado por localidad:")
print(df_mapa)
print(f"\n✅ {len(df_mapa)} localidades con datos completos")

## 4. Análisis Exploratorio de Datos (EDA)

In [ ]:
# Gráfico 1: Top 10 localidades con más animales identificados
fig = px.bar(
    df_mapa.nlargest(10, 'total_animales'),
    x='localidad',
    y='total_animales',
    title='Top 10 Localidades con Más Animales Identificados con Microchip',
    labels={'localidad': 'Localidad', 'total_animales': 'Número de Animales'},
    color='total_animales',
    color_continuous_scale='Viridis'
)
fig.update_layout(xaxis_tickangle=-45, height=500)
fig.show()

In [ ]:
# Gráfico 2: Distribución por especie
especie_counts = df['especie'].value_counts()
fig = px.pie(
    values=especie_counts.values,
    names=especie_counts.index,
    title='Distribución de Animales por Especie',
    hole=0.4
)
fig.update_traces(textposition='inside', textinfo='percent+label')
fig.show()

In [ ]:
# Gráfico 3: Distribución por tamaño
tamano_counts = df['tamano_animal'].value_counts()
fig = px.bar(
    x=tamano_counts.index,
    y=tamano_counts.values,
    title='Distribución de Animales por Tamaño',
    labels={'x': 'Tamaño', 'y': 'Cantidad'},
    color=tamano_counts.values,
    color_continuous_scale='Blues'
)
fig.show()

In [ ]:
# Gráfico 4: Caninos vs Felinos por localidad
df_especie = df_mapa[['localidad', 'total_caninos', 'total_felinos']].nlargest(10, 'total_animales')

fig = go.Figure()
fig.add_trace(go.Bar(x=df_especie['localidad'], y=df_especie['total_caninos'], name='Caninos'))
fig.add_trace(go.Bar(x=df_especie['localidad'], y=df_especie['total_felinos'], name='Felinos'))

fig.update_layout(
    title='Comparación Caninos vs Felinos por Localidad (Top 10)',
    xaxis_title='Localidad',
    yaxis_title='Cantidad',
    barmode='group',
    xaxis_tickangle=-45,
    height=500
)
fig.show()

In [ ]:
# Gráfico 5: Animales potencialmente peligrosos
peligroso_counts = df['potencialmente_peligroso'].value_counts()
fig = px.pie(
    values=peligroso_counts.values,
    names=peligroso_counts.index,
    title='Distribución de Animales Potencialmente Peligrosos',
    color_discrete_sequence=['#00CC96', '#EF553B']
)
fig.update_traces(textposition='inside', textinfo='percent+label+value')
fig.show()

In [ ]:
# Tabla resumen de estadísticas
print("\n📊 RESUMEN ESTADÍSTICO GENERAL\n")
print("="*60)
print(f"Total de animales identificados: {len(df):,}")
print(f"Total de localidades: {df['localidad_territorializacion'].nunique()}")
print(f"Total de caninos: {len(df[df['especie']=='CANINO']):,}")
print(f"Total de felinos: {len(df[df['especie']=='FELINO']):,}")
print(f"Animales potencialmente peligrosos: {len(df[df['potencialmente_peligroso']=='SI']):,}")
print(f"Porcentaje de animales peligrosos: {(len(df[df['potencialmente_peligroso']=='SI'])/len(df)*100):.2f}%")
print("="*60)

## 5. LOAD - Carga de Datos a Base de Datos SQLite

In [ ]:
# Conectar a la base de datos
db_path = '../database/geo_bog.db'

# Crear conexión
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print("✅ Conexión a base de datos establecida")

In [ ]:
# Crear tabla para animales si no existe
cursor.execute('''
CREATE TABLE IF NOT EXISTS animales_microchip (
    microchip TEXT PRIMARY KEY,
    especie TEXT,
    sexo TEXT,
    tamano TEXT,
    potencialmente_peligroso TEXT,
    localidad TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

# Crear tabla para estadísticas por localidad
cursor.execute('''
CREATE TABLE IF NOT EXISTS estadisticas_localidad (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    localidad TEXT UNIQUE,
    latitud REAL,
    longitud REAL,
    total_animales INTEGER,
    total_caninos INTEGER,
    total_felinos INTEGER,
    total_peligrosos INTEGER,
    especie_predominante TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')

conn.commit()
print("✅ Tablas creadas correctamente")

In [ ]:
# Cargar datos de animales
df_db = df[['microchip_animal', 'especie', 'sexo_animal', 'tamano_animal', 
            'potencialmente_peligroso', 'localidad_territorializacion']].copy()
df_db.columns = ['microchip', 'especie', 'sexo', 'tamano', 'potencialmente_peligroso', 'localidad']

# Insertar datos (reemplazar si existen)
df_db.to_sql('animales_microchip', conn, if_exists='replace', index=False)
print(f"✅ {len(df_db)} registros de animales cargados")

# Cargar estadísticas por localidad
df_mapa_db = df_mapa[['localidad', 'latitud', 'longitud', 'total_animales', 
                       'total_caninos', 'total_felinos', 'total_peligrosos', 'especie_predominante']]
df_mapa_db.to_sql('estadisticas_localidad', conn, if_exists='replace', index=False)
print(f"✅ {len(df_mapa_db)} registros de estadísticas cargados")

conn.close()
print("\n✅ ETL completado - Datos cargados a la base de datos")

## 6. Mapa Interactivo con Folium

In [ ]:
# Crear mapa base centrado en Bogotá
bogota_coords = [4.6097, -74.0817]
mapa = folium.Map(
    location=bogota_coords,
    zoom_start=11,
    tiles='OpenStreetMap'
)

# Agregar capa de tiles alternativa
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(mapa)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Dark').add_to(mapa)

# Normalizar valores para el tamaño de los marcadores
min_animals = df_mapa['total_animales'].min()
max_animals = df_mapa['total_animales'].max()

# Agregar marcadores para cada localidad
for idx, row in df_mapa.iterrows():
    # Calcular tamaño del marcador basado en cantidad de animales
    size = 10 + (row['total_animales'] - min_animals) / (max_animals - min_animals) * 30
    
    # Determinar color basado en la cantidad
    if row['total_animales'] > 2000:
        color = 'red'
    elif row['total_animales'] > 1000:
        color = 'orange'
    elif row['total_animales'] > 500:
        color = 'blue'
    else:
        color = 'green'
    
    # Crear popup con información
    popup_html = f"""
    <div style="font-family: Arial; width: 250px;">
        <h4 style="color: #2C3E50; margin-bottom: 10px;">{row['localidad']}</h4>
        <hr style="margin: 5px 0;">
        <b>📊 Total Animales:</b> {row['total_animales']:,}<br>
        <b>🐕 Caninos:</b> {row['total_caninos']:,}<br>
        <b>🐱 Felinos:</b> {row['total_felinos']:,}<br>
        <b>⚠️ Peligrosos:</b> {row['total_peligrosos']:,}<br>
        <b>🏆 Predominante:</b> {row['especie_predominante']}<br>
        <hr style="margin: 5px 0;">
        <small>📍 Lat: {row['latitud']:.4f}, Lon: {row['longitud']:.4f}</small>
    </div>
    """
    
    # Agregar marcador circular
    folium.CircleMarker(
        location=[row['latitud'], row['longitud']],
        radius=size,
        popup=folium.Popup(popup_html, max_width=300),
        color=color,
        fillColor=color,
        fillOpacity=0.6,
        weight=2
    ).add_to(mapa)
    
    # Agregar etiqueta con el nombre de la localidad
    folium.Marker(
        location=[row['latitud'], row['longitud']],
        icon=folium.DivIcon(html=f"""
            <div style="
                font-size: 9px; 
                color: black;
                font-weight: bold;
                text-shadow: 1px 1px 2px white;
                white-space: nowrap;
            ">{row['localidad']}</div>
        """)
    ).add_to(mapa)

# Agregar leyenda
legend_html = '''
<div style="
    position: fixed; 
    bottom: 50px; 
    left: 50px; 
    width: 220px; 
    background-color: white; 
    border: 2px solid grey; 
    z-index: 9999; 
    font-size: 14px;
    padding: 10px;
    border-radius: 5px;
">
    <h4 style="margin: 0 0 10px 0;">🗺️ Leyenda</h4>
    <p style="margin: 5px 0;"><span style="color: red;">●</span> &gt; 2000 animales</p>
    <p style="margin: 5px 0;"><span style="color: orange;">●</span> 1000 - 2000 animales</p>
    <p style="margin: 5px 0;"><span style="color: blue;">●</span> 500 - 1000 animales</p>
    <p style="margin: 5px 0;"><span style="color: green;">●</span> &lt; 500 animales</p>
</div>
'''
mapa.get_root().html.add_child(folium.Element(legend_html))

# Agregar control de capas
folium.LayerControl().add_to(mapa)

# Agregar herramientas adicionales
plugins.Fullscreen().add_to(mapa)
plugins.MeasureControl(position='topleft').add_to(mapa)

# Guardar mapa
output_path = '../web/static/mapa_animales_bogota.html'
mapa.save(output_path)
print(f"✅ Mapa guardado en: {output_path}")

# Mostrar mapa
mapa

## 7. Mapa de Calor (Heatmap)

In [ ]:
# Crear mapa de calor
mapa_calor = folium.Map(
    location=bogota_coords,
    zoom_start=11,
    tiles='CartoDB positron'
)

# Preparar datos para el heatmap (repetir coordenadas según cantidad)
heat_data = []
for idx, row in df_mapa.iterrows():
    # Agregar punto con peso según cantidad de animales
    heat_data.append([row['latitud'], row['longitud'], row['total_animales']/100])

# Agregar capa de calor
plugins.HeatMap(
    heat_data,
    min_opacity=0.4,
    radius=25,
    blur=30,
    gradient={
        0.0: 'blue',
        0.4: 'lime',
        0.6: 'yellow',
        0.8: 'orange',
        1.0: 'red'
    }
).add_to(mapa_calor)

# Agregar título
title_html = '''
<div style="
    position: fixed; 
    top: 10px; 
    left: 50%; 
    transform: translateX(-50%);
    width: 600px; 
    background-color: white; 
    border: 2px solid grey; 
    z-index: 9999; 
    font-size: 16px;
    padding: 10px;
    border-radius: 5px;
    text-align: center;
">
    <h3 style="margin: 0;">🔥 Mapa de Calor - Densidad de Animales con Microchip por Localidad</h3>
</div>
'''
mapa_calor.get_root().html.add_child(folium.Element(title_html))

# Guardar mapa de calor
heatmap_path = '../web/static/mapa_calor_animales_bogota.html'
mapa_calor.save(heatmap_path)
print(f"✅ Mapa de calor guardado en: {heatmap_path}")

mapa_calor

## 8. Exportar Datos Procesados

In [ ]:
# Guardar datos procesados
output_csv = '../data/processed/animales_microchip_procesado.csv'
df.to_csv(output_csv, index=False, encoding='utf-8')
print(f"✅ Datos completos guardados en: {output_csv}")

# Guardar estadísticas por localidad
output_stats = '../data/processed/estadisticas_por_localidad.csv'
df_mapa.to_csv(output_stats, index=False, encoding='utf-8')
print(f"✅ Estadísticas por localidad guardadas en: {output_stats}")

# Guardar en formato JSON para uso en web
output_json = '../web/static/datos_localidades.json'
df_mapa.to_json(output_json, orient='records', indent=2)
print(f"✅ Datos JSON guardados en: {output_json}")

## 9. Conclusiones y Hallazgos

### Hallazgos Principales:

1. **Distribución Geográfica**: Las localidades con mayor registro de animales con microchip se concentran en zonas con mayor desarrollo urbano.

2. **Especies**: Se observa una predominancia de caninos sobre felinos en todas las localidades.

3. **Animales Peligrosos**: El porcentaje de animales potencialmente peligrosos es bajo en relación al total.

4. **Tendencias por Localidad**: Las localidades con mayor identificación de animales corresponden a zonas con mayor población y desarrollo socioeconómico.

### Recomendaciones:

1. **Expansión de Programas**: Fortalecer programas de identificación en localidades con menor cobertura.

2. **Control de Animales Peligrosos**: Mantener seguimiento especial en localidades con mayor concentración de animales potencialmente peligrosos.

3. **Educación**: Implementar campañas educativas sobre la importancia del microchip en todas las localidades.

4. **Monitoreo Continuo**: Establecer sistemas de monitoreo periódico para identificar tendencias y patrones.

In [ ]:
print("\n" + "="*70)
print("🎉 ANÁLISIS COMPLETADO EXITOSAMENTE")
print("="*70)
print("\n📁 Archivos generados:")
print("   ✓ Base de datos SQLite")
print("   ✓ Mapa interactivo por localidades")
print("   ✓ Mapa de calor (heatmap)")
print("   ✓ Datos procesados en CSV")
print("   ✓ Estadísticas por localidad")
print("   ✓ Datos en formato JSON")
print("\n🗺️ Los mapas se pueden visualizar en el navegador web")
print("="*70)